<a href="https://colab.research.google.com/github/yzhou007/llama-2-from-scratch/blob/main/llama_2_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# model.py

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Optional

In [3]:
@dataclass
class ModelArgs:
  dim: int = 4096
  n_layers: int = 32
  n_heads: int = 32
  n_kv_heads: Optional[int] = None # grouped query attention
  vocab_size: int = -1 # This will be set when we load the tokenizer
  multiple_of: int = 256
  ffn_dim_multiplier: Optional[float] = None
  norm_eps: float = 1e-5

  # Needed for KV cache
  max_batch_size: int = 32
  max_seq_len: int = 2048

  device: str = None


## RMS Norm

In [4]:
class RMSNorm(nn.Module):
  def __init__(self, dim: int, eps: float = 1e-8):
    super().__init__()

    self.eps = eps
    self.scale = nn.Parameter(torch.ones(dim))

  def forward(self, x: torch.Tensor):
    # (batch, seq_len, dim)
    rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True))
    return self.scale * x / (rms + self.eps)

## ROPE

In [5]:
def precompute_theta_pos_frequencies(head_dim: int, seq_len: int, device: str, theta: float = 10000.0):
    # As written in the paragraph 3.2.2 of the paper
    # >> In order to generalize our results in 2D to any xi ∈ Rd where **d is even**, [...]
    assert head_dim % 2 == 0, "Dimension must be divisible by 2"
    # Build the theta parameter
    # According to the formula theta_i = 10000^(-2(i-1)/dim) for i = [1, 2, ... dim/2]
    # Shape: (Head_Dim / 2)
    theta_numerator = torch.arange(0, head_dim, 2).float()
    # Shape: (Head_Dim / 2)
    theta = 1.0 / (theta ** (theta_numerator / head_dim)).to(device) # (Dim / 2)
    # Construct the positions (the "m" parameter)
    # Shape: (Seq_Len)
    m = torch.arange(seq_len, device=device)
    # Multiply each theta by each position using the outer product.
    # Shape: (Seq_Len) outer_product* (Head_Dim / 2) -> (Seq_Len, Head_Dim / 2)
    freqs = torch.outer(m, theta).float()
    # We can compute complex numbers in the polar form c = R * exp(m * theta), where R = 1 as follows:
    # (Seq_Len, Head_Dim / 2) -> (Seq_Len, Head_Dim / 2)
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex

def apply_rotary_embeddings(x: torch.Tensor, freqs_complex: torch.Tensor, device: str):
    # Separate the last dimension pairs of two values, representing the real and imaginary parts of the complex number
    # Two consecutive values will become a single complex number
    # (B, Seq_Len, H, Head_Dim) -> (B, Seq_Len, H, Head_Dim/2)
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    # Reshape the freqs_complex tensor to match the shape of the x_complex tensor. So we need to add the batch dimension and the head dimension
    # (Seq_Len, Head_Dim/2) --> (1, Seq_Len, 1, Head_Dim/2)
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)
    # Multiply each complex number in the x_complex tensor by the corresponding complex number in the freqs_complex tensor
    # Which results in the rotation of the complex number as shown in the Figure 1 of the paper
    # (B, Seq_Len, H, Head_Dim/2) * (1, Seq_Len, 1, Head_Dim/2) = (B, Seq_Len, H, Head_Dim/2)
    x_rotated = x_complex * freqs_complex
    # Convert the complex number back to the real number
    # (B, Seq_Len, H, Head_Dim/2) -> (B, Seq_Len, H, Head_Dim/2, 2)
    x_out = torch.view_as_real(x_rotated)
    # (B, Seq_Len, H, Head_Dim/2, 2) -> (B, Seq_Len, H, Head_Dim)
    x_out = x_out.reshape(*x.shape)
    return x_out.type_as(x).to(device)

## Transformer

In [6]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    batch_size, seq_len, n_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    return (
        # (B, Seq_Len, N_KV_Heads, 1, Head_Dim)
        x[:, :, :, None, :]
        # (B, Seq_Len, N_KV_Heads, N_Rep, Head_Dim)
        .expand(batch_size, seq_len, n_kv_heads, n_rep, head_dim)
        # (B, Seq_Len, N_KV_Heads * N_Rep, Head_Dim)
        .reshape(batch_size, seq_len, n_kv_heads * n_rep, head_dim)
    )

## SelfAttention

In [7]:
# Aultiple grouped-query attnetion with KV cache.
# Only works for inference
class SelfAttention(nn.Module):
  def __init__(self, args: ModelArgs):
    super().__init__()

    self.n_kv_heads = args.n_kv_heads if args.n_kv_heads is not None else args.n_heads
    self.n_heads = args.n_heads
    self.n_rep = self.n_heads//self.n_kv_heads

    self.dim = args.dim
    self.head_dim = self.dim // self.n_heads

    self.wq = nn.Linear(self.dim, self.n_heads * self.head_dim, bias=False)
    self.wk = nn.Linear(self.dim, self.n_kv_heads * self.head_dim, bias=False)
    self.wv = nn.Linear(self.dim, self.n_kv_heads * self.head_dim, bias=False)
    self.wo = nn.Linear(self.dim, self.dim, bias=False)

    self.cache_k = torch.zeros(args.max_batch_size, args.max_seq_len, self.n_kv_heads, self.head_dim)
    self.cache_v = torch.zeros(args.max_batch_size, args.max_seq_len, self.n_kv_heads, self.head_dim)

  def forward(self, x: torch.Tensor, start_pos:int, freqs_complex: torch.Tensor):
    # (batch, 1, dim)
    # Note that seq_len = 1 since we only pass 1 token at each step with KV cache
    batch_size, seq_len, _ = x.shape

    # Projection:
    xq = self.wq(x) # (batch, 1, n_heads * head_dim)
    xk = self.wk(x) # (batch, 1, n_kv_heads * head_dim)
    xv = self.wv(x) # (batch, 1, n_kv_heads * head_dim)

    # Reshape into different heads: (batch, 1, dim) -> (batch, 1, n_heads/n_kv_heads, head_dim)
    xq = xq.view(xq.shape[0], xq.shape[1], self.n_heads, self.head_dim)
    xk = xk.view(xk.shape[0], xk.shape[1], self.n_kv_heads, self.head_dim)
    xv = xv.view(xv.shape[0], xv.shape[1], self.n_kv_heads, self.head_dim)

    # Apply ROPE
    # Note that we only apply to key and value when we need to calculate attentions)
    xq = apply_rotary_embeddings(xq, freqs_complex, device=x.device)
    xk = apply_rotary_embeddings(xk, freqs_complex, device=x.device)

    # Append latest key and value to the cache
    self.cache_k[:batch_size, start_pos : start_pos+seq_len] = xk
    self.cache_v[:batch_size, start_pos : start_pos+seq_len] = xv

    # Retrive all the cached keys and values
    # (batch, kv_cache, n_kv_heads, head_dim)
    keys = self.cache_k[:batch_size, 0:start_pos+seq_len]
    values = self.cache_v[:batch_size, 0:start_pos+seq_len]

    # Repeat the heads for k and v to match the one for q
    keys = repeat_kv(keys, self.n_rep)
    values = repeat_kv(values, self.n_rep)

    # Calculate attention scores
    # (batch, seq_len, n_heads, head_dim) -> (batch, n_heads, seq_len, head_dim)
    xq = xq.transpose(1, 2)

    # (batch, kv_cache, n_heads, head_dim) -> (batch, n_heads, kv_cache, head_dim)
    keys = keys.transpose(1, 2)
    values = values.transpose(1, 2)

    # (batch, n_heads, seq_len, kv_cache)
    attention_scores = torch.matmul(xq, keys.transpose(-2, -1)) / math.sqrt(self.head_dim)
    # (batch, n_heads, seq_len, kv_cache)
    attention_scores = F.softmax(attention_scores.float(), dim=-1).type_as(xq)

    # (batch, n_heads, seq_len, head_dim)
    output = torch.matmul(attention_scores, values)
    # (batch, seq_len, n_heads * head_dim)
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

    # attention output projection
    # (batch, seq_len, dim)
    output = self.wo(output)
    return output

In [8]:
class FeedForward(nn.Module):
    def __init__(
        self,
        args: ModelArgs
    ):
        super().__init__()

        hidden_dim = 4 * args.dim
        hidden_dim = int(2 * hidden_dim / 3)
        if args.ffn_dim_multiplier is not None:
            hidden_dim = int(args.ffn_dim_multiplier * hidden_dim)
        # Round the hidden_dim to the nearest multiple of the multiple_of parameter
        hidden_dim = args.multiple_of * ((hidden_dim + args.multiple_of - 1) // args.multiple_of)

        self.w1 = nn.Linear(args.dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(args.dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, args.dim, bias=False)

    def forward(self, x: torch.Tensor):
        # (B, Seq_Len, Dim) --> (B, Seq_Len, Hidden_Dim)
        swish = F.silu(self.w1(x))
        # (B, Seq_Len, Dim) --> (B, Seq_Len, Hidden_Dim)
        x_V = self.w3(x)
        # (B, Seq_Len, Hidden_Dim) * (B, Seq_Len, Hidden_Dim) --> (B, Seq_Len, Hidden_Dim)
        x = swish * x_V
        # (B, Seq_Len, Hidden_Dim) --> (B, Seq_Len, Dim)
        x = self.w2(x)
        return x

## Encoder Block

In [9]:
class EncoderBlock(nn.Module):

  def __init__(self, args: ModelArgs):
    super().__init__()

    self.n_heads = args.n_heads
    self.dim = args.dim
    self.head_dim = args.dim // args.n_heads

    self.attention = SelfAttention(args)
    self.feed_forward = FeedForward(args)

    # Norm before self-attention
    self.attention_norm = RMSNorm(self.head_dim)

    # Norm before feedforward
    self.ffn_norm = RMSNorm(self.head_dim)

  def forward(self, x: torch.Tensor, freqs_complex: torch.Tensor):
    h = x + self.attention(self.attention_norm(x), freqs_complex)

    output = h + self.feed_forward(self.ffn_norm(h))
    return output



In [10]:
class Transformer(nn.Module):
  def __init__(self, args: ModelArgs) -> None:
    super().__init__()

    self.args = args
    self.vocab_size = args.vocab_size
    self.n_layers = args.n_layers
    self.tok_embeddings = nn.Embedding(self.vocab_size, args.dim)

    self.layers = nn.ModuleList()

    for _ in range(args.n_layers):
      self.layers.append(EncoderBlock(args))

    self.norm = RMSNorm(args.dim, eps=args.norm_eps)
    self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

    self.freqs_complex = precompute_theta_pos_frequencies(self.args.dim // self.args.n_heads, self.args.max_seq_len * 2, device=self.args.device)

  def forward(self, tokens: torch.Tensor, start_pos: int):
    # (batch_size, seq_len)
    batch_size, seq_len = tokens.shape
    assert seq_len == 1 # Only 1 token at 1 time in inference due to KV cache

    # (batch_size, seq_len) -> (batch_size, seq_len, dim)
    h = self.tok_embeddings(tokens)

    freqs_complex = self.freqs_complex[start_pos : start_pos + seq_len]

    for layer in self.layers:
      h = layer(h, freqs_complex)

    h = self.norm(h)
    logits = self.output(h).float()

    return logits

# inference.py

In [11]:
from typing import Optional
import torch
import time
from pathlib import Path
import json
from sentencepiece import SentencePieceProcessor
from tqdm import tqdm

In [12]:
class LLaMA:

    def __init__(self, model: Transformer, tokenizer: SentencePieceProcessor, model_args: ModelArgs):
        self.model = model
        self.tokenizer = tokenizer
        self.args = model_args

    @staticmethod
    def build(checkpoints_dir: str, tokenizer_path: str, load_model: bool, max_seq_len: int, max_batch_size: int, device: str):
        prev_time = time.time()
        if load_model:
            checkpoints = sorted(Path(checkpoints_dir).glob("*.pth"))
            assert len(checkpoints) > 0, f"no checkpoint files found in {checkpoints_dir}"
            ckpt_path = checkpoints[0]
            print(f'Loading checkpoint "{ckpt_path}"')
            checkpoint = torch.load(ckpt_path, map_location="cpu")
            print(f"Loaded checkpoint in {time.time() - prev_time:.2f}s")
            prev_time = time.time()
        with open(Path(checkpoints_dir) / "params.json", "r") as f:
            params = json.loads(f.read())

        model_args: ModelArgs = ModelArgs(
            max_seq_len=max_seq_len,
            max_batch_size=max_batch_size,
            device=device,
            **params
        )

        tokenizer = SentencePieceProcessor()
        tokenizer.load(tokenizer_path)
        model_args.vocab_size = tokenizer.vocab_size()

        if device == "cuda":
            torch.set_default_tensor_type(torch.cuda.HalfTensor)
        else:
            torch.set_default_tensor_type(torch.BFloat16Tensor)

        model = Transformer(model_args).to(device)

        if load_model:
            # The only unmatched key in the checkpoint is rope.freqs. Remove it
            del checkpoint['rope.freqs']
            model.load_state_dict(checkpoint, strict=True)
            print(f"Loaded state dict in {time.time() - prev_time:.2f}s")

        return LLaMA(model, tokenizer, model_args)

In [ ]:
if __name__ == '__main__':
    torch.manual_seed(0)

    allow_cuda = False
    device = 'cuda' if torch.cuda.is_available() and allow_cuda else 'cpu'

    # prompts = [
    #     "Simply put, the theory of relativity states that ",
    #     "If Google was an Italian company founded in Milan, it would",
    #     # Few shot promt
    #     """Translate English to French:

    #     sea otter => loutre de mer
    #     peppermint => menthe poivrée
    #     plush girafe => girafe peluche
    #     cheese =>""",
    #     # Zero shot prompt
    #     """Tell me if the following person is actually Doraemon disguised as human:
    #     Name: Umar Jamil
    #     Decision:
    #     """
    # ]


    model = LLaMA.build(
        checkpoints_dir='llama-2-7b/',
        tokenizer_path='tokenizer.model',
        load_model=True,
        max_seq_len=1024,
        max_batch_size=3, #len(prompts),
        device=device
    )
